# EDA dan Visualisasi Interaktif

Melakukan Exploratory Data Analysis (EDA) dan menghasilkan minimal 6 visualisasi interaktif dan 5 insight berdasarkan dataset Indonesia Affordable Housing.

In [ ]:
import polars as pl
import plotly.express as px

CLEAN_DATA_PATH = '../data/indonesia_housing_data_clean.parquet'

# Load data bersih lazily dan konversi ke Pandas untuk plotting (atau collect() ke dalam memori)
df = pl.scan_parquet(CLEAN_DATA_PATH).collect()
df_pd = df.to_pandas()

## 1. Analisis Spasial: Total Unit Properti per Provinsi

In [ ]:
units_by_prov = df_pd.groupby('province')['total_units'].sum().reset_index()
fig1 = px.bar(units_by_prov, x='province', y='total_units', title='Total Unit Properti per Provinsi')
fig1.show()

**Insight 1:** Visualisasi menunjukkan distribusi pembangunan unit perumahan di berbagai provinsi, mengindikasikan provinsi mana yang memiliki fokus pengembangan perumahan terjangkau terbesar.

## 2. Hubungan Harga dan Luas Tanah

In [ ]:
df_sample = df_pd.sample(n=10000, random_state=42) if len(df_pd) > 10000 else df_pd
fig2 = px.scatter(df_sample, x='land_area_m2', y='price_idr', color='subsidy_status', title='Harga Properti vs Luas Tanah')
fig2.show()

**Insight 2:** Terdapat korelasi positif antara luas tanah dan harga rumah. Rumah subsidi cenderung mengelompok pada area harga dan luas yang lebih rendah sesuai regulasi.

## 3. Proporsi Status Subsidi

In [ ]:
subsidy_counts = df_pd['subsidy_status'].value_counts().reset_index()
subsidy_counts.columns = ['subsidy_status', 'count']
fig3 = px.pie(subsidy_counts, names='subsidy_status', values='count', title='Proporsi Rumah Subsidi vs Komersial')
fig3.show()

**Insight 3:** Porsi antara rumah subsidi dan komersial menunjukkan sejauh mana ketersediaan rumah yang ditujukan untuk masyarakat berpenghasilan rendah dalam data ini.

## 4. Distribusi Harga Properti

In [ ]:
fig4 = px.histogram(df_sample, x='price_idr', nbins=50, title='Distribusi Harga Properti')
fig4.show()

**Insight 4:** Sebagian besar harga properti terpusat di rentang harga tertentu, kemungkinan besar di batas maksimal harga rumah subsidi yang ditetapkan pemerintah.

## 5. Rata-rata Harga Berdasarkan Jumlah Kamar Tidur

In [ ]:
price_by_bed = df_pd.groupby('bedrooms')['price_idr'].mean().reset_index()
fig5 = px.bar(price_by_bed, x='bedrooms', y='price_idr', title='Rata-rata Harga Berdasarkan Jumlah Kamar Tidur')
fig5.show()

**Insight 5:** Secara umum, rata-rata harga properti meningkat seiring dengan bertambahnya jumlah kamar tidur, mencerminkan kebutuhan ruang yang lebih besar.

## 6. Peta Sebaran Properti (Geospasial)

In [ ]:
df_map = df_pd.dropna(subset=['latitude', 'longitude'])
df_map_sample = df_map.sample(n=5000, random_state=42) if len(df_map) > 5000 else df_map
fig6 = px.scatter_mapbox(df_map_sample, lat='latitude', lon='longitude', color='price_idr',
                         size='land_area_m2', hover_name='housing_name',
                         mapbox_style='carto-positron', zoom=3, title='Peta Sebaran Lokasi Properti')
fig6.show()

**Insight Tambahan:** Visualisasi peta menunjukkan aglomerasi perumahan yang terpusat di pulau Jawa dan kota-kota besar lainnya, sejalan dengan tingkat kepadatan penduduk.